# SBND -> ICARUS conditional constraint

How much of the ICARUS systematic uncertainty survives the SBND measurement,
in which directions, and why -- computed from PROfit's PROplot output through
`mutools.profit`.

**Conventions.** Everything is *fractional*: a shift $\delta$ means
$N_i \to N_i(1+\delta_i)$, and the fractional statistical variance of a
Poisson bin with expectation $N$ is $1/N$.

**The model.** Nuisance knobs $\theta\sim\mathcal N(0,I)$; SBND fractional
residual $r = A\theta + \varepsilon$ with $\varepsilon\sim\mathcal N(0,S)$,
$S=\mathrm{diag}(1/N)$; ICARUS shift $B\theta$. The surviving ICARUS
covariance is the Gaussian conditional

$$R \;=\; C_{yy} - C_{yx}\bigl(C_{xx} + S/\alpha\bigr)^{-1}C_{xy},$$

equivalently $R = B\Lambda B^{\mathsf T}$ with
$\Lambda = (I + \alpha A^{\mathsf T}S^{-1}A)^{-1}$ the post-SBND knob
covariance -- so the conditional *is* the profiled linear fit.

**Both ICARUS runs are conditioned together.** Run 2 and Run 4 are separate
blocks of the covariance, but one near detector constrains them jointly;
conditioning them in a single step keeps the Run 2 / Run 4 correlations
instead of discarding them.

**Input.** Requires the `Covariance` directory, which PROfit writes only when
asked:

```
PROfit -x xml/contours/contours_recoe.xml -t contours_recoe -o covar -n 8 \
       plot --with-covar
```

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from mutools import profit
from mutools.plotting import use_style
from mutools.plotting.helpers import mark_axis

use_style('rootlike')

INPUT = Path('/Users/mueller/Projects/spineosc/runs/legacy_root/contours_recoe_covar_PROplot.root')

NEAR = 'SBND'
FAR = ('ICARUSRun2', 'ICARUSRun4')

# Fixed series order, checked for colourblind separation. Assigned in order
# and never cycled; the yellow slot is skipped because it is too low-contrast
# for a thin line on white.
SERIES_COLORS = ('#2a78d6', '#eb6834', '#1baf7a', '#4a3aa7', '#e87ba4', '#008300')

## What the file holds

The loader infers the covariance bin ordering by rebuilding the file's own
`collapsed_total_frac_cov`, so a successful load is already a consistency
check. Channels may be binned differently -- `contours_recoe` bins `ccqelike`
in 16 reconstructed-energy bins and `nproton` in 14 -- which is why bin counts
are per channel rather than global.

The fit variable must be 1D: for a 2D channel PROfit writes only the x
projection per subchannel, which cannot fill the covariance space (loading a
`contours_base` file fails with exactly that diagnostic).

In [ ]:
pp = profit.ProPlot(INPUT)

print('detectors:  ', pp.detectors)
print('channels:   ', pp.channels)
print('bins/channel:', pp.channel_bins)
print('components: ', len(pp.components))
print(f'full space: {pp.cv.shape[0]} bins -> collapsed {pp.cv_collapsed.shape[0]}'
      f' ({pp.detector_width()} per detector)')

In [ ]:
def block_boundaries(proplot, detectors):
    """Channel boundaries and labels within a concatenated detector block.

    Returns the bin index at which each channel starts (for divider lines)
    and a (centre, label) pair per channel for annotation.
    """
    starts, labels = [], []
    position = 0
    for detector in detectors:
        for mode, channel in proplot.channels:
            width = proplot.channel_bins[(mode, channel)]
            starts.append(position)
            labels.append((position + 0.5 * width, f'{detector}\n{channel}'))
            position += width
    return starts, labels


def draw_blocks(ax, proplot, detectors, label_y=0.92):
    """Mark the channel blocks of a concatenated detector block on an axis."""
    starts, labels = block_boundaries(proplot, detectors)
    for start in starts[1:]:
        ax.axvline(start, color='0.6', lw=0.8, ls=':', zorder=1)
    for centre, text in labels:
        ax.text(centre, label_y, text, transform=ax.get_xaxis_transform(),
                ha='center', va='top', fontsize=7, color='0.35')

## The constraint

`constraint()` builds the fractional covariance from the selected components,
adds SBND's Poisson statistics, and conditions the far block on it.

In [ ]:
result = profit.constraint(pp, nd_detector=NEAR, fd_detector=FAR)

print(f'far block: {result.residual.shape[0]} bins')
print(f'prior       mean frac error: {result.prior_error.mean():.4f}')
print(f'constrained mean frac error: {result.constrained_error.mean():.4f}')
print(f'mean reduction:              {result.reduction.mean():.3f}')

fig, ax = plt.subplots(figsize=(9, 4.5))
bins = np.arange(result.residual.shape[0])
ax.step(bins, result.prior_error, where='mid', color='0.35', lw=1.6,
        label='Before constraint')
ax.step(bins, result.constrained_error, where='mid', color=SERIES_COLORS[0],
        lw=1.8, label='After SBND constraint')
draw_blocks(ax, pp, FAR)
ax.set_xlabel('ICARUS bin (Run 2 then Run 4)')
ax.set_ylabel('Fractional uncertainty')
ax.set_ylim(bottom=0)
ax.legend(frameon=True, framealpha=0.85, edgecolor='none', fontsize=8,
          loc='lower left')
mark_axis(ax, 'SBN Simulation')

## The systematic families

Fractional component covariances add linearly, so any `select`/`exclude`
subset is an exact partition of the *covariance*. The **constraint** is not
linear in that partition, though -- conditioning inverts
$C_{xx}+S/\alpha$ -- so a family in isolation and a family removed from the
total answer two different questions, and neither set of numbers sums to the
total. Both are reported below.

The families partition every component in the file, and the cell asserts it:
a systematic added to the XML later cannot silently fall outside the
breakdown.

What the numbers show on this file is worth stating plainly: the constraint
works on the families that are *shared* between the detectors and does
nothing at all to the rest.

- **Cross section, flux and G4** are correlated near-to-far, and SBND removes
  most of them (isolation ratios 0.15, 0.14, 0.20).
- **Detector, POT normalisation and MC stat** come out with ratio **exactly
  1.000** -- untouched. Each of those components is declared per detector and
  run (`ChargeScale_icarus_Run2`, `ToroidCalibration_sbnd_Run1`, ...), so
  there is no near/far correlation for the conditional to exploit. No amount
  of SBND data constrains them; they are simply what is left over.
- **CORSIKA** is a cosmic-rate normalisation on SBND alone, so its ICARUS
  prior is exactly zero. It enters only as near-detector noise, and removing
  it improves the total constraint by 4e-5 -- real, but negligible here.

In [ ]:
FAMILIES = {
    'Cross section': r'^(GENIEReWeight|ZExpPCA|CCQETemplate|MECq0q3|QEInterference)',
    'Flux': r'_Flux$',
    'Detector': r'^(ChargeScale|SCE|TriggerEmulation|WireMod|DENT|Gain)',
    'POT norm': r'^(ToroidCalibration|ProtonMiss)',
    'G4': r'^reinteractions',
    'CORSIKA': r'cos:',
    'MC stat': r'mcstat',
}

# The families must tile the component list exactly: no component counted
# twice, none left out.
assignment = {}
for label, pattern in FAMILIES.items():
    for name in profit.match_components(pp.component_names, pattern, label):
        assert name not in assignment, f'{name} matches {label} and {assignment[name]}'
        assignment[name] = label
missing = set(pp.component_names) - set(assignment)
assert not missing, f'components in no family: {sorted(missing)}'
print(f'{len(assignment)} components in {len(FAMILIES)} families')

rows = []
for label, pattern in FAMILIES.items():
    alone = profit.constraint(pp, nd_detector=NEAR, fd_detector=FAR, select=pattern)
    removed = profit.constraint(pp, nd_detector=NEAR, fd_detector=FAR, exclude=pattern)
    rows.append((label, sum(v == label for v in assignment.values()),
                 alone.prior_error.mean(), alone.constrained_error.mean(),
                 removed.constrained_error.mean()))

print(f'\n{"family":15s} {"n":>3s} {"alone prior":>12s} {"alone constr":>13s}'
      f' {"total w/o it":>13s}')
print(f'{"TOTAL":15s} {len(assignment):3d} {result.prior_error.mean():12.4f}'
      f' {result.constrained_error.mean():13.4f} {"-":>13s}')
for label, count, prior, constrained, without in sorted(rows, key=lambda r: -r[3]):
    print(f'{label:15s} {count:3d} {prior:12.4f} {constrained:13.4f} {without:13.4f}')

### Each family on its own, before and after

How large a family is on the far detector, and how much of it SBND removes.
A family whose two bars are nearly equal is one the near detector cannot
see; a family whose second bar collapses is one it constrains away.

In [ ]:
labels = [row[0] for row in rows]
y = np.arange(len(labels))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(y - 0.2, [row[2] for row in rows], height=0.38, color='0.75',
        label='Alone, before constraint')
ax.barh(y + 0.2, [row[3] for row in rows], height=0.38, color=SERIES_COLORS[0],
        label='Alone, after SBND constraint')
ax.set_yticks(y)
ax.set_yticklabels(labels)
ax.invert_yaxis()
ax.set_xlabel('Mean fractional uncertainty on ICARUS')
ax.legend(frameon=True, framealpha=0.85, edgecolor='none', fontsize=8,
          loc='lower right')
mark_axis(ax, 'SBN Simulation')

### What the total owes to each family

Each curve is the full budget with one family removed, so the gap to `Total`
is what that family contributes *after* the constraint has done its work.
Removing CORSIKA can lower the curve: it is near-detector noise, so dropping
it sharpens the constraint rather than removing far-detector uncertainty.

In [ ]:
shown = ['Cross section', 'Flux', 'Detector', 'POT norm', 'CORSIKA']

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.step(bins, result.constrained_error, where='mid', color='0.35', lw=1.8,
        label='Total')
for color, label in zip(SERIES_COLORS, shown):
    res = profit.constraint(pp, nd_detector=NEAR, fd_detector=FAR,
                            exclude=FAMILIES[label])
    ax.step(bins, res.constrained_error, where='mid', color=color, lw=1.4,
            label=f'without {label}')

draw_blocks(ax, pp, FAR)
ax.set_xlabel('ICARUS bin (Run 2 then Run 4)')
ax.set_ylabel('Fractional uncertainty\nafter SBND constraint')
ax.set_ylim(bottom=0)
ax.legend(frameon=True, framealpha=0.85, edgecolor='none', fontsize=7,
          loc='lower left', ncol=2)

### Detector systematics, broken down

Since the family as a whole survives the constraint untouched, this breakdown
is about *size*: which detector effects dominate what is left on ICARUS.

Each curve is that systematic alone. Two asymmetries to keep in mind, both
from the XML rather than from the physics: `SCE` carries only the two ICARUS
entries and `DENT` only the SBND one (the SBND SCE and Gain entries are
commented out), and a SBND-only systematic has zero ICARUS prior by
construction -- DENT's curve sits on zero for that reason, not because it is
small.

In [ ]:
DETECTOR_SOURCES = {
    'Charge scale': r'^ChargeScale',
    'Space charge': r'^SCE',
    'Trigger emulation': r'^TriggerEmulation',
    'WireMod x-thetaXW': r'^WireModxThetaXW',
    'WireMod YZ': r'^WireModYZ',
    'DENT': r'^DENT',
}

fig, ax = plt.subplots(figsize=(9, 4.5))
detector_only = profit.constraint(pp, nd_detector=NEAR, fd_detector=FAR,
                                  select=FAMILIES['Detector'])
ax.step(bins, detector_only.constrained_error, where='mid', color='0.35',
        lw=1.8, label='All detector systematics')

for color, (label, pattern) in zip(SERIES_COLORS, DETECTOR_SOURCES.items()):
    names = profit.match_components(pp.component_names, pattern, label)
    res = profit.constraint(pp, nd_detector=NEAR, fd_detector=FAR, select=pattern)
    ax.step(bins, res.constrained_error, where='mid', color=color, lw=1.4,
            label=f'{label} ({len(names)})')
    print(f'{label:20s} {len(names)} component(s), mean constrained '
          f'{res.constrained_error.mean():.4f}')

draw_blocks(ax, pp, FAR)
ax.set_xlabel('ICARUS bin (Run 2 then Run 4)')
ax.set_ylabel('Fractional uncertainty\nafter SBND constraint')
# Headroom for the block labels, and the legend below the axes: seven series
# do not fit inside the frame without covering the curves.
ax.set_ylim(0, detector_only.constrained_error.max() * 1.35)
ax.legend(frameon=False, fontsize=7, loc='upper center',
          bbox_to_anchor=(0.5, -0.18), ncol=4)

## Exposure scan

$\alpha$ scales the SBND statistics, $S\to S/\alpha$. As
$\alpha\to\infty$ the posterior tends to the projector onto knob directions
SBND cannot see at all, so the limiting curve is a **degeneracy floor**, not a
statistical one: bins already sitting on it want a new observable, not more
POT.

In [ ]:
scan = {rf'$\alpha$ = {a:g}': profit.constraint(pp, nd_detector=NEAR, fd_detector=FAR,
                                                alpha=a)
        for a in (0.25, 1.0, 4.0)}
scan[r'$\alpha \to \infty$'] = profit.constraint(pp, nd_detector=NEAR, fd_detector=FAR,
                                                 statistics=False)

fig, ax = plt.subplots(figsize=(9, 4.5))
for color, (label, res) in zip(SERIES_COLORS, scan.items()):
    ax.step(bins, res.constrained_error, where='mid', color=color, lw=1.6, label=label)

draw_blocks(ax, pp, FAR)
ax.set_xlabel('ICARUS bin (Run 2 then Run 4)')
ax.set_ylabel('Fractional uncertainty\nafter SBND constraint')
ax.set_ylim(bottom=0)
ax.legend(frameon=True, framealpha=0.85, edgecolor='none', fontsize=8,
          loc='lower left')

## What survives, and in what shapes

The residual is low-rank: a handful of modes carry the surviving uncertainty.
Each $\sqrt{\lambda_a}\,v_a$ is a one-sigma fractional shift of the ICARUS
prediction, and the participation ratio
$n_{\mathrm{eff}} = (\sum\lambda)^2/\sum\lambda^2$ counts the modes that
matter.

In [ ]:
values, vectors, participation = profit.eigenmodes(result.residual)
print(f'participation ratio: {participation:.2f} effective modes')
print('leading eigenvalues: ', values[:5].round(6))

fig, ax = plt.subplots(figsize=(9, 4.5))
for index, color in enumerate(SERIES_COLORS[:3]):
    shape = np.sqrt(values[index]) * vectors[:, index]
    ax.step(bins, shape, where='mid', color=color, lw=1.6,
            label=rf'mode {index}, $\sqrt{{\lambda}}$ = {np.sqrt(values[index]):.3f}')

ax.axhline(0.0, color='0.6', lw=0.8)
draw_blocks(ax, pp, FAR)
ax.set_xlabel('ICARUS bin (Run 2 then Run 4)')
ax.set_ylabel(r'One-sigma fractional shift')
ax.legend(frameon=True, framealpha=0.85, edgecolor='none', fontsize=8,
          loc='lower left')

## Per-knob budget: what SBND could constrain vs what it does

For a single knob, Sherman-Morrison gives the ceiling
$\varepsilon^{\max}_k = g_k/(1+g_k)$ with visibility
$g_k = a_k^{\mathsf T}S^{-1}a_k$ -- the $\chi^2$ with which SBND alone
distinguishes a $1\sigma$ excursion. The achieved efficiency in the joint fit
is $\varepsilon_k = 1-\Lambda_{kk}$, and the gap between them is constraint
lost to *degeneracy* with other knobs rather than to statistics.

Building responses eigen-decomposes every selected component, so this cell is
the slow one.

In [ ]:
model = profit.build_responses(pp, nd_detector=NEAR, fd_detector=FAR)
post = profit.posterior(model, pp.fractional_stats(NEAR))
print(f'{model.n_knobs} effective knobs from {len(model.captured)} components')

order = np.argsort(post.efficiency)[::-1][:20]
y = np.arange(len(order))

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(y, post.efficiency_ceiling[order], color='0.85', label='Statistical ceiling')
ax.barh(y, post.efficiency[order], height=0.55, color=SERIES_COLORS[0],
        label='Achieved in joint fit')
ax.set_yticks(y)
ax.set_yticklabels([model.labels[i] for i in order], fontsize=6)
ax.invert_yaxis()
ax.set_xlabel('Constraint efficiency')
ax.set_xlim(0, 1)
ax.legend(frameon=False, fontsize=8, loc='lower right')